Bygger en CSV fil med server objekt, att populera ett datacenter med.

In [247]:
import csv

In [248]:
csv_data = """Web-Server-01,192.168.1.10,16
Web-Server-02,192.168.1.11,16
Web-Server-03,192.168.1.12,8
App-Server-01,192.168.1.13,32
App-Server-02,192.168.1.14,32
App-Server-03,192.168.1.15,16
DB-Master-01,192.168.1.16,64
DB-Slave-01,192.168.1.17,32
DB-Slave-02,192.168.1.18,32
Backup-Server,192.168.1.19,8
Proxy-01,192.168.1.20,4
Proxy-02,192.168.1.21,4
Auth-Server,192.168.1.22,8
DNS-Server-01,192.168.1.23,4
DNS-Server-02,192.168.1.24,4
Monitoring-01,192.168.1.25,16
Storage-01,192.168.1.26,12
Storage-02,192.168.1.27,12
Dev-Server-01,192.168.1.28,8
Test-Server-01,192.168.1.29,4"""

with open("server.csv", "w", encoding="utf-8") as file:
    file.write(csv_data.strip())

print("En seed file för att populera datacenter med 20 enheter har skapats")

En seed file för att populera datacenter med 20 enheter har skapats


Ett första bygge av Klasser med arv

In [249]:
# Parent/Child klasser

class Hardware:
    def __init__(self, name: str, ip_address: str):
        self.name = name
        self.ip_address = ip_address
        self.status = "OK" 
        
class Server(Hardware):
    def __init__(self, name: str, ip_address: str, cpu_cores: int):
        super().__init__(name, ip_address)
        self.cpu_cores = cpu_cores

print("Parent Klass Hardware och Child  Klass Server har nu definierats i en första version")

Parent Klass Hardware och Child  Klass Server har nu definierats i en första version


Nu ska vi läsa in csv filen i internminnet i en så kallad lista (inventory[]), för att därefter populera upp datacenter

# inläsning av CSV fil

In [250]:
inventory = []

with open("server.csv", "r", encoding ="utf-8") as file:
    reader = csv.reader(file)
    for row in reader:
        server_object = Server(name=row[0], ip_address=row[1], cpu_cores=int(row[2]))
        inventory.append(server_object)
        
print(f"totalt antal inlästa server objekt är {len(inventory)}")
 

totalt antal inlästa server objekt är 20


Nu ska vi testa att läsa från inventory listan i minnet

In [251]:
print("läser vad som finns på rad två i listan (index startar från 0)")
print(f"namnet är:{inventory[1].name} IP är:{inventory[1].ip_address} antal cpu är {inventory[1].cpu_cores}")
print("\nläser innehållet i de tre första raderna i listan")
for srv in inventory[:3]:
    print(f"namnet är: {srv.name} IP är: {srv.ip_address} antal cpu är: {srv.cpu_cores}")        



läser vad som finns på rad två i listan (index startar från 0)
namnet är:Web-Server-02 IP är:192.168.1.11 antal cpu är 16

läser innehållet i de tre första raderna i listan
namnet är: Web-Server-01 IP är: 192.168.1.10 antal cpu är: 16
namnet är: Web-Server-02 IP är: 192.168.1.11 antal cpu är: 16
namnet är: Web-Server-03 IP är: 192.168.1.12 antal cpu är: 8


Nu testas förändring av värden i listan

In [252]:
print(f"vi ändrar antal cpu i Web-Server-3 från {inventory[2].cpu_cores} till {2*inventory[2].cpu_cores}")

# --- 4. FÖRÄNDRA ETT OBJEKT I LISTAN ---
# Vi hämtar databasservern (index 6 i listan) och simulera ett fel
ws_3 = inventory[2]
ws_3.cpu_cores = 2*inventory[2].cpu_cores 

print(f"{ws_3.name} har nu {ws_3.cpu_cores} cpu kärnor ")


for rader in inventory[:3]:
    rader.status = "Warning"
    print (rader.status, rader.name, rader.cpu_cores)   

for rader in inventory[:]:
    if rader.status == "Warning":
        rader.status = "CRITICAL"
        print(rader.status, rader.name, rader.cpu_cores)  
    
 

vi ändrar antal cpu i Web-Server-3 från 8 till 16
Web-Server-03 har nu 16 cpu kärnor 
Warning Web-Server-01 16
Warning Web-Server-02 16
Warning Web-Server-03 16
CRITICAL Web-Server-01 16
CRITICAL Web-Server-02 16
CRITICAL Web-Server-03 16


Nu ska vi ID tagga alla server objekt 
Målet är att varje server skall få en ID tagg (IT-01, IT-02 osv) 

In [253]:
id_counter = 1

class Hardware:
    def __init__(self, name: str, ip_address: str):
        global id_counter
        # varje NY instans får ett unikt ID när __init__ körs
        self.id_tag = f"ID-{id_counter}"
        id_counter += 1
        
        self.name = name
        self.ip_address = ip_address
        self.status = "OK"
        
        
class Server(Hardware):
    def __init__(self, name: str, ip_address: str, cpu_cores: int):
        # super() anropar Hardware.__init__ som sätter id_tag, name, ip_adress och status
        super().__init__(name, ip_address)
        self.cpu_cores = cpu_cores
        

inventory = []

with open("server.csv", "r", encoding="utf-8") as file:
    reader = csv.reader(file)
    
    for row in reader:
        # När server() anropas körs __init__ så att id_tag skapas automatiskt
        server_object = Server(name=row[0], ip_address=row[1], cpu_cores=int(row[2]))
        inventory.append(server_object)
    

for srv in inventory:
    print(f"{srv.id_tag} {srv.name}")
    
print("Nu har alla servrar id taggats via en ny attribut och metod i Parent Class hardware")

ID-1 Web-Server-01
ID-2 Web-Server-02
ID-3 Web-Server-03
ID-4 App-Server-01
ID-5 App-Server-02
ID-6 App-Server-03
ID-7 DB-Master-01
ID-8 DB-Slave-01
ID-9 DB-Slave-02
ID-10 Backup-Server
ID-11 Proxy-01
ID-12 Proxy-02
ID-13 Auth-Server
ID-14 DNS-Server-01
ID-15 DNS-Server-02
ID-16 Monitoring-01
ID-17 Storage-01
ID-18 Storage-02
ID-19 Dev-Server-01
ID-20 Test-Server-01
Nu har alla servrar id taggats via en ny attribut och metod i Parent Class hardware


Nu ska vi göra samma sak men genom att arbeta med listan i minnet och 
utöka med en label som heter id_tag, och inte bara instansera om alla objekt 

In [254]:
inventory = []
with open("server.csv", "r", encoding="utf-8") as file:
    reader = csv.reader(file)
    for row in reader:    
        server_object = Server(name=row[0], ip_address=row[1], cpu_cores=int(row[2]))
        inventory.append(server_object)

print("Testar att skriva ut rad 3 för att verifiera att en lista skapats i minnet")
print (inventory[2].name)


Testar att skriva ut rad 3 för att verifiera att en lista skapats i minnet
Web-Server-03


Nu ska vi lägga till en klassvariabel i listan med de 
instanserade servrarna som redan ligger i minnet
Det gör vi genom att sätta en klassvariabel DIREKT 
på Parent Class Hardware genom hardware.datacenter_site = "DC_STOCKHOLM_ZONE_1"
Nu finns denna egenskap, dvs label sitenamn för varje server, AUTOMATISKT tillagd i samtliga serverobjekt i minnet 

In [255]:
print("Vi sätter den nya klassvariabeln datacenter_site till DC_STOCKHOLM_ZONE_1")
Hardware.datacenter_site = "DC_STOCKHOLM_ZONE_1"

print("Testar att skriva ut rad 3 för att verifiera att klassvariabeln skapats i minnet")
print (inventory[2].datacenter_site)

Vi sätter den nya klassvariabeln datacenter_site till DC_STOCKHOLM_ZONE_1
Testar att skriva ut rad 3 för att verifiera att klassvariabeln skapats i minnet
DC_STOCKHOLM_ZONE_1


Nu ska vi lägga till id_tagg till listan med de 
instanserade servrarna som redan ligger i minnet
Detta genom en for loop som säkerhetstaggar 
BEFINTLIG population, alltså jag instasierar INTE en ny.
Vi stegar igenom listan i minnet och sätter en ny instansvariabel (.security_tag)

In [256]:
# 1. LOOPA IGENOM ALLA SERVRAR OCH SÄTT TAGGEN
for server in inventory:
    if server.cpu_cores >= 32:
        server.security_tag = "High_Risk"
    elif server.cpu_cores >= 16:
        server.security_tag = "Medium_Risk"
    else:
        server.security_tag = "Low_Risk"

# 2. SKRIV UT DE 8 FÖRSTA SERVRARNA
print(f"{'NAMN':<15} | {'CPU':<5} | {'SÄKERHETSTAGG'}")
print("-" * 40)

for srv in inventory[:8]:
    print(f"{srv.name:<15} | {srv.cpu_cores:<5} | {srv.security_tag}")

NAMN            | CPU   | SÄKERHETSTAGG
----------------------------------------
Web-Server-01   | 16    | Medium_Risk
Web-Server-02   | 16    | Medium_Risk
Web-Server-03   | 8     | Low_Risk
App-Server-01   | 32    | High_Risk
App-Server-02   | 32    | High_Risk
App-Server-03   | 16    | Medium_Risk
DB-Master-01    | 64    | High_Risk
DB-Slave-01     | 32    | High_Risk


I detta exempel baseras security_tag på hårdvarukapacitet (CPU), medan warning_tag baseras på en helt annan regel (t.ex. om servern har för lite RAM-minne för sin CPU-kraft):
BÅDA TAGGARNA NU INUTI Server class

In [257]:
class Hardware:

    def __init__(
        self, name: str, ip_address: str, power_watts: int, asset_tag: str = ""
    ):
        self.name = name
        self.ip_address = ip_address
        self.power_watts = power_watts
        self.asset_tag = asset_tag
        self.status = "OK"


class Server(Hardware):

    def __init__(
        self,
        name: str,
        ip_address: str,
        power_watts: int,
        cpu_cores: int,
        ram_gb: int,
        asset_tag: str = "",
    ):
        # 1. Anropa parent-klassen med super()
        super().__init__(name, ip_address, power_watts, asset_tag)

        # 2. Grundläggande specifikationer
        self.cpu_cores = cpu_cores
        self.ram_gb = ram_gb

        # 3. TAGG 1: Security Tag (Baserat på CPU-kärnor / Riskprofil)
        if self.cpu_cores >= 32:
            self.security_tag = "High_Risk"
        elif self.cpu_cores >= 16:
            self.security_tag = "Medium_Risk"
        else:
            self.security_tag = "Low_Risk"

        # 4. TAGG 2: Warning Tag (Baserat på resursbalans / Varningar)
        # Regel: Om vi har mycket CPU (>=16) men lite RAM (<32GB) varnar vi för flaskhals
        if self.cpu_cores >= 16 and self.ram_gb < 32:
            self.warning_tag = "RAM_BOTTLENECK_WARNING"
        elif self.power_watts > 600:
            self.warning_tag = "HIGH_POWER_WARNING"
        else:
            self.warning_tag = "NO_WARNING"


# =====================================================================
# TESTA ATT SKAPA OCH SKRIVA UT SERVRARNA
# =====================================================================

inventory = [
    Server(
        "Web-Server-01",
        "192.168.1.10",
        power_watts=400,
        cpu_cores=16,
        ram_gb=16,
        asset_tag="HW-1",
    ),  # RAM-flaskhals!
    Server(
        "App-Server-01",
        "192.168.1.20",
        power_watts=750,
        cpu_cores=32,
        ram_gb=128,
        asset_tag="HW-2",
    ),  # Hög strömförbrukning!
    Server(
        "Test-Server-01",
        "192.168.1.30",
        power_watts=200,
        cpu_cores=4,
        ram_gb=16,
        asset_tag="HW-3",
    ),  # Helt OK
]

print(
    f"{'NAMN':<15} | {'SECURITY TAG':<13} | {'WARNING TAG':<23} | {'POWER'}"
)
print("-" * 65)

for srv in inventory:
    print(
        f"{srv.name:<15} | {srv.security_tag:<13} | {srv.warning_tag:<23} | {srv.power_watts}W"
    )

NAMN            | SECURITY TAG  | WARNING TAG             | POWER
-----------------------------------------------------------------
Web-Server-01   | Medium_Risk   | RAM_BOTTLENECK_WARNING  | 400W
App-Server-01   | High_Risk     | HIGH_POWER_WARNING      | 750W
Test-Server-01  | Low_Risk      | NO_WARNING              | 200W


Problemet här är att vi behöver instansera om server objekten vid tillägg av taggar (labels/kolumner). Därför behöver vi frångå hårda taggar som kräver ominstansering och i stället titta på
dynamiska set med taggar på levande objekt. 


men...

Att kasta in fridrivande textsträngar i en set() eller ändra godtyckliga variabler under drift kallas i branschen för "Magic Strings" och "Unstructured State". Det är en känd källa till buggar eftersom stavfel (t.ex. "GOLD_SLA" vs "gold_sla") inte upptäcks av Python och det finns ingen validering.

För ett stabilt, professionellt och lättunderhållet DCIM-system finns det två betydligt bättre och renare arkitekturmönster.

Det Bästa Alternativet: Beräknade Egenskaper (@property)
Istället för att spara en sekundär tagg eller variabel i minnet och behöva "uppdatera" den manuellt när saker ändras, beräknar vi statusen dynamiskt när någon ber om den.

Varför detta är överlägset:
Single Source of Truth: Du har bara en sanningskälla (förhållandet mellan cpu_cores och ram_gb).

Noll synkroniseringsfel: Du kan aldrig glömma att uppdatera taggen – den räknas ut exakt i samma stund du frågar efter den.

Ingen minnesbelastning: Inga extra variabler eller listor sparas på objektet.


Kodexempel: Rent & Stabilt med @property

OBSERVERA EXTRA NOGA @property dekoratorn


In [258]:
class Hardware:

    def __init__(
        self, 
        name: str, 
        ip_address: str, 
        power_watts: int, 
        asset_tag: str = ""
    ):
        self.name = name
        self.ip_address = ip_address
        self.power_watts = power_watts
        self.asset_tag = asset_tag
        self.status = "OK"


class Server(Hardware):

    def __init__(
        self,
        name: str,
        ip_address: str,
        power_watts: int,
        cpu_cores: int,
        ram_gb: int,
        asset_tag: str = "",
    ):
        super().__init__(name, ip_address, power_watts, asset_tag)
        self.cpu_cores = cpu_cores
        self.ram_gb = ram_gb

    # --- BERÄKNADE EGENSKAPER (COMPUTED PROPERTIES) ---

    @property
    def security_level(self) -> str:
        """Räknar ut säkerhetsklass dynamiskt baserat på hårdvarans tillstånd."""
        if self.cpu_cores >= 32:
            return "HIGH_RISK"
        elif self.cpu_cores >= 16:
            return "MEDIUM_RISK"
        return "LOW_RISK"

    @property
    def has_bottleneck(self) -> bool:
        """Kollar om servern har för lite RAM i förhållande till sina CPU-kärnor."""
        return self.cpu_cores >= 16 and self.ram_gb < 32


# =====================================================================
# TESTA I DRIFT
# =====================================================================

# 1. Vi skapar 3 servrar i drift
inventory = [
    Server("Web-01", "192.168.1.10", 400, cpu_cores=16, ram_gb=16, asset_tag="HW-1"),
    Server("App-01", "192.168.1.20", 750, cpu_cores=32, ram_gb=128, asset_tag="HW-2"),
    Server("DB-01", "192.168.1.30", 200, cpu_cores=8, ram_gb=32, asset_tag="HW-3"),
]

# 2. Skriv ut rapporten
print(
    f"{'NAMN':<10} | {'CPU':<4} | {'RAM':<5} | {'RISKKLASS':<12} | {'FLASKHALS?'}"
)
print("-" * 55)

for srv in inventory:
    # Vi anropar .security_level och .has_bottleneck som om de vore vanliga variabler!
    print(
        f"{srv.name:<10} | {srv.cpu_cores:<4} | {srv.ram_gb:<5} | {srv.security_level:<12} | {srv.has_bottleneck}"
    )

NAMN       | CPU  | RAM   | RISKKLASS    | FLASKHALS?
-------------------------------------------------------
Web-01     | 16   | 16    | MEDIUM_RISK  | True
App-01     | 32   | 128   | HIGH_RISK    | False
DB-01      | 8    | 32    | LOW_RISK     | False


In [259]:
# Vi gör en hårdvaruuppgradering i minnet
srv = inventory[0]  # Web-01
srv.ram_gb = 64  # Nytt RAM-minne

# När vi läser av har flaskhalsen försvunnit 
# AUTOMATISKT utan att vi uppdaterade några taggar!
print(f"Web-01 Flaskhals efter uppgradering: {srv.has_bottleneck}") 
# Outputs: False

Web-01 Flaskhals efter uppgradering: False


"Att manuellt lägga till och ändra lösa tagg-variabler i efterhand skapar risk för osynkroniserad data och är svårt att felsöka. I mitt DCIM-system använder jag därför Computed Properties (@property).

Det gör att egenskaper som riskklass och prestandaflaskhalsar beräknas direkt utifrån objektets grunddata när de efterfrågas. Om en server får mer minne eller ändrar status i drift, speglas det direkt i beräkningen utan att vi behöver köra uppdateringsskript eller riskera att sekundära variabler blir inaktuella."

# Del 2. Sensorer och övervakning 

Steg 1: Objektinteraktion (Separation of Concerns)
I OOP är det en viktig princip att varje klass har sitt eget ansvarsområde:

En Server håller koll på sina specifikationer (CPU, RAM, IP-adress).

En TemperatureSensor håller koll på fysisk temperatur och mäter en specifik enhet.

För att sensorn ska kunna övervaka servern skickar vi med serverobjektet som en referens till sensorn när den skapas:

In [260]:
# Här skapar vi Sensor klassen

class TemperatureSensor:

    def __init__(self, sensor_id: str, target_hardware):
        self.sensor_id = sensor_id
        # Vi sparar en referens till HÅRDVARUOBJEKTET direkt inuti sensorn!
        self.target_hardware = target_hardware
        self.current_temp = 22.0  # Starttemperatur i Celsius

    def read_temperature(self) -> float:
        """Returnerar den nuvarande temperaturen."""
        # self.current_temp = self.current_temp+1
        return self.current_temp
    
    def get_sensor_status(self):
        temp = self.read_temperature()
        return f"Sensor[{self.sensor_id}] på [{self.target_hardware.name}]: [{temp}] °C"

    def update_temperature(self, new_temp: float):
        """Uppdaterar temperaturen OCH ändrar status på servern om det blir för varmt."""
        self.current_temp = new_temp

        # Här läser vi av temperaturen och kollar tröskelvärdena:
        if self.current_temp >= 75.0:
            self.target_hardware.status = "CRITICAL"
        elif self.current_temp >= 50.0:
            self.target_hardware.status = "WARNING"
        else:
            self.target_hardware.status = "OK"



Här skapar vi servern som ska övervakas via sensorn
som vi också skapar här.

In [261]:
# 1. Skapa servern
my_server = Server(
    "Web-Server-01", "192.168.1.10", power_watts=400, cpu_cores=16, ram_gb=32
)

# 2. Skapa sensorn och peka ut servern som target!
sensor_1 = TemperatureSensor(
    sensor_id="TEMP-001", target_hardware=my_server
)

# 3. Sensorn kan nu komma åt serverns namn via sin referens:
print(
    f"Sensorn {sensor_1.sensor_id} övervakar just nu: {sensor_1.target_hardware.name}"
)

print(sensor_1.read_temperature())

sensor_1.read_temperature()
sensor_1.get_sensor_status()


Sensorn TEMP-001 övervakar just nu: Web-Server-01
22.0


'Sensor[TEMP-001] på [Web-Server-01]: [22.0] °C'

Nytt test, nu med ändring av temperatur genom metoden update_temperature

In [262]:
# Skapa server och sensor
server = Server("Web-01", "192.168.1.10", 400, 16, 32)
sensor = TemperatureSensor("TEMP-01", server)

print("Status före:", server.status)  # Borde vara OK

# Ändra temperaturen till 82 grader
sensor.update_temperature(82.0)

print("Ny temp via read_temperature():", sensor.read_temperature())
print("Status efter värmebölja:", server.status)  # Borde ha blivit CRITICAL!

Status före: OK
Ny temp via read_temperature(): 82.0
Status efter värmebölja: CRITICAL


Testar nu med en random snurra för olika temperatur och därmed olika reaktion

In [263]:
import random
import time
from datetime import datetime

# =====================================================================
# 1. KÖR EN SIMULERINGS-LOOP (5 MÄTNINGAR)
# =====================================================================

print("--- STARTAR AUTOMATISK TEMPERATURSIMULERING ---")

# Vi kör 5 slumpmässiga mätningar efter varandra
for i in range(1, 6):
    # Generera ett slumpmässigt värde mellan 25.0 °C och 90.0 °C (avrundat till 1 decimal)
    simulated_temp = round(random.uniform(25.0, 90.0), 1)

    # Skicka in det slumpade värdet till din uppdateringsmetod!
    sensor_1.update_temperature(simulated_temp)

    # Skriv ut vad som hände vid denna mätning
    print(
        f"Mätning {i}: Temp={simulated_temp:<4} °C | Server status: {my_server.status}"
    )

    # Lite paustid (0.5 sekunder) så att tidsstämplarna hinner skilja sig åt
    time.sleep(0.5)

print("\n--- SIMULERING KLAR ---")

--- STARTAR AUTOMATISK TEMPERATURSIMULERING ---
Mätning 1: Temp=45.6 °C | Server status: OK
Mätning 2: Temp=72.2 °C | Server status: WARNING
Mätning 3: Temp=47.6 °C | Server status: OK
Mätning 4: Temp=44.2 °C | Server status: OK
Mätning 5: Temp=45.5 °C | Server status: OK

--- SIMULERING KLAR ---


Steg 3 i Del 2: Enkel Fil-loggning
Det tredje och sista steget i Del 2 handlar om att spara larm till en textfil på disken.

När sensorn upptäcker en hög temperatur räcker det inte att ändra statusen i minnet – vi vill också att den skriver en rad i filen dcim_events.log.

För att lägga till text i en fil utan att radera det som redan står där använder vi läget "a" (append):

In [264]:
class TemperatureSensor:

    def __init__(self, sensor_id: str, target_hardware):
        self.sensor_id = sensor_id
        self.target_hardware = target_hardware
        self.current_temp = 22.0

    def read_temperature(self) -> float:
        return self.current_temp

    def update_temperature(self, new_temp: float):
        self.current_temp = new_temp

        # Om det blir för varmt: ändra status OCH skriv till fil
        if self.current_temp >= 75.0:
            self.target_hardware.status = "CRITICAL"

            # Enkel fil-skrivning
            with open("dcim_events.log", "a") as f:
                f.write(
                    f"LARM: {self.target_hardware.name} har kritisk temp: {self.current_temp} °C\n"
                )

        elif self.current_temp >= 50.0:
            self.target_hardware.status = "WARNING"

            with open("dcim_events.log", "a") as f:
                f.write(
                    f"VARNING: {self.target_hardware.name} har hög temp: {self.current_temp} °C\n"
                )

        else:
            self.target_hardware.status = "OK"

In [265]:
server = Server("Web-01", "192.168.1.10", 400, 16, 32)
sensor = TemperatureSensor("TEMP-01", server)

sensor.update_temperature(82.0)

with open("dcim_events.log", "r") as f:
    print(f.read())

LARM: Web-01 har kritisk temp: 82.0 °C
LARM: Web-01 har kritisk temp: 82.0 °C
LARM: Web-01 har kritisk temp: 82.0 °C
LARM: Web-01 har kritisk temp: 82.0 °C
LARM: Web-01 har kritisk temp: 82.0 °C
2026-10-02 16:45:17 - ERROR - Web-01 är CRITICAL: 80.0°C
LARM: Web-01 har kritisk temp: 82.0 °C
2026-10-02 17:25:57 - ERROR - Web-01 är CRITICAL: 80.0°C
2026-10-02 17:25:57 - ERROR - Web-01 är CRITICAL: 85.0°C
2026-10-02 17:25:57 - INFO - Web-01 är i normal drift: 35.0°C
LARM: Web-01 har kritisk temp: 82.0 °C



In [266]:
import os

print("Filen sparades i denna mapp:")
print(os.getcwd())

# Kollar om filen faktiskt finns i minnet/disken just nu
print("Finns filen på disken?", os.path.exists("dcim_events.log"))

Filen sparades i denna mapp:
c:\Users\mansk\my_dcim
Finns filen på disken? True


Det fungerade sådär.
Därför kör vi nu påpå Pythons riktiga, inbyggda logging-modul!

Att manuellt öppna filer med open() fungerar, men i professionella Python-system använder man logging. Det ger dig automatiska tidsstämplar, loggnivåer (INFO, WARNING, ERROR) och hanterar filskrivningen i bakgrunden utan krångel.

💡 Hur logging fungerar (Grunden)
Istället för with open() konfigurerar vi loggern en gång högst upp i koden:

In [267]:
import logging

# 1. Konfigurera hur loggern ska bete sig
logging.basicConfig(
    filename="dcim_events.log",  # Filen som ska skapas
    level=logging.INFO,  # Lägsta nivå som ska sparas (INFO, WARNING, ERROR)
    format="%(asctime)s - %(levelname)s - %(message)s",  # Format på raden
    datefmt="%Y-%m-%d %H:%M:%S",  # Snygg tidsstämpel
)




Info kring logging

Nedan prövar vi logging modulen i vårat Temperatur Sensor scenario.

In [268]:
class TemperatureSensor:

    def __init__(self, sensor_id: str, target_hardware):
        self.sensor_id = sensor_id
        self.target_hardware = target_hardware
        self.current_temp = 22.0

    def update_temperature(self, new_temp: float):
        self.current_temp = new_temp

        if self.current_temp >= 75.0:
            self.target_hardware.status = "CRITICAL"
            # Skickar direkt till loggfilen som en ERROR
            logging.error(
                f"{self.target_hardware.name} är CRITICAL: {self.current_temp}°C"
            )

        elif self.current_temp >= 50.0:
            self.target_hardware.status = "WARNING"
            # Skickar direkt till loggfilen som en WARNING
            logging.warning(
                f"{self.target_hardware.name} har WARNING: {self.current_temp}°C"
            )

        else:
            self.target_hardware.status = "OK"
            
            
    

In [269]:
server2 = Server("Web-01", "192.168.1.10", 400, 16, 32)
sensor2 = TemperatureSensor("TEMP-01", server2)

sensor2.update_temperature(80.0)

In [270]:
with open("dcim_events.log", "r") as f:
    print(f.read())

LARM: Web-01 har kritisk temp: 82.0 °C
LARM: Web-01 har kritisk temp: 82.0 °C
LARM: Web-01 har kritisk temp: 82.0 °C
LARM: Web-01 har kritisk temp: 82.0 °C
LARM: Web-01 har kritisk temp: 82.0 °C
2026-10-02 16:45:17 - ERROR - Web-01 är CRITICAL: 80.0°C
LARM: Web-01 har kritisk temp: 82.0 °C
2026-10-02 17:25:57 - ERROR - Web-01 är CRITICAL: 80.0°C
2026-10-02 17:25:57 - ERROR - Web-01 är CRITICAL: 85.0°C
2026-10-02 17:25:57 - INFO - Web-01 är i normal drift: 35.0°C
LARM: Web-01 har kritisk temp: 82.0 °C
2026-10-02 17:27:06 - ERROR - Web-01 är CRITICAL: 80.0°C



 Ser du hur stor skillnaden är?
De första raderna var från din gamla with open()-lösning. Den sista raden:
2026-10-02 16:45:17 - ERROR - Web-01 är CRITICAL: 80.0°C
...visar exakt hur kraftfull logging-modulen är. Tidsstämpel, loggnivå (ERROR) och meddelande skapades helt automatiskt!


Övning 2: Logga när läget blir normalt (INFO)
I ett riktigt övervakningssystem vill vi inte bara logga när saker går sönder (ERROR / WARNING), utan även när en server återhämtar sig och blir grön/OK igen.

För normala händelser använder vi loggnivån logging.info().

In [271]:
class TemperatureSensor:

    def __init__(self, sensor_id: str, target_hardware):
        self.sensor_id = sensor_id
        self.target_hardware = target_hardware
        self.current_temp = 22.0

    def update_temperature(self, new_temp: float):
        self.current_temp = new_temp

        if self.current_temp >= 75.0:
            self.target_hardware.status = "CRITICAL"
            # Skickar direkt till loggfilen som en ERROR
            logging.error(
                f"{self.target_hardware.name} är CRITICAL: {self.current_temp}°C"
            )

        elif self.current_temp >= 50.0:
            self.target_hardware.status = "WARNING"
            # Skickar direkt till loggfilen som en WARNING
            logging.warning(
                f"{self.target_hardware.name} har WARNING: {self.current_temp}°C"
            )

        else:
            logging.info(
                f"{self.target_hardware.name} är i normal drift: {self.current_temp}°C"
            )

Höjer och sänker temperaturen för larm och återställning

In [272]:
server3 = Server("Web-01", "192.168.1.10", 400, 16, 32)
sensor3 = TemperatureSensor("TEMP-01", server3)

# 1. Höj temperaturen så det blir LARM
sensor3.update_temperature(85.0)
# 2. Sänk temperaturen så återställs LARM 
sensor3.update_temperature(35.0)

In [273]:
with open("dcim_events.log", "r") as f:
    print(f.read())

LARM: Web-01 har kritisk temp: 82.0 °C
LARM: Web-01 har kritisk temp: 82.0 °C
LARM: Web-01 har kritisk temp: 82.0 °C
LARM: Web-01 har kritisk temp: 82.0 °C
LARM: Web-01 har kritisk temp: 82.0 °C
2026-10-02 16:45:17 - ERROR - Web-01 är CRITICAL: 80.0°C
LARM: Web-01 har kritisk temp: 82.0 °C
2026-10-02 17:25:57 - ERROR - Web-01 är CRITICAL: 80.0°C
2026-10-02 17:25:57 - ERROR - Web-01 är CRITICAL: 85.0°C
2026-10-02 17:25:57 - INFO - Web-01 är i normal drift: 35.0°C
LARM: Web-01 har kritisk temp: 82.0 °C
2026-10-02 17:27:06 - ERROR - Web-01 är CRITICAL: 80.0°C
2026-10-02 17:27:06 - ERROR - Web-01 är CRITICAL: 85.0°C
2026-10-02 17:27:06 - INFO - Web-01 är i normal drift: 35.0°C



"Att använda Pythons logging-modul ger oss en strukturerad och konfigurerbar hantering av händelser. Vi får automatisk tidsstämpling, standardiserade formateringar och olika loggnivåer (INFO, WARNING, ERROR).

I en produktionsmiljö gör detta att vi enkelt kan filtrera loggar – till exempel söka efter enbart akuta ERROR-händelser – utan att behöva bygga egen strängformatering."

SAMMANFATTNING LARMNING OCH LOGGNING

✅ Objektinteraktion: Sensorn har en referens till servern och kan ändra dess status i minnet.

✅ Strukturerad Loggning: Alla varningar och återhämtningar skrivs automatiskt till fil med tidsstämpel via logging.